In [1]:
import pandas as pd

from datetime import datetime
from datetime import date
from dateutil.relativedelta import relativedelta

In [2]:
df = pd.read_excel(
    "dados.xlsx",
    sheet_name=0,
    header=1,          # usa a 2ª linha como cabeçalho real
    dtype={"Class. Por idade": str},  # evita converter "1" em número se quiser
)


## Limpando / Organizando - Dataset

In [4]:
# Posteriormente transformar todo esse codigo em uma funcaozinha (PIPELINE)
df.columns = (
    df.columns.astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
# Renomeado as colunas
df = df.rename(columns={
    "Nome": "nome",
    "Class. Por idade": "classificacao_idade",
    "Data nascimento": "data_nascimento",
    "Data 1ª consulta": "data_primeira_consulta",
    "Tempo entre início dos sintomas e primeira consulta": "intervalo_sintomas_primeira_consulta",
    "Sexo": "sexo",
    "Tipo": "tipo",
    "Perda de peso": "perda_peso",
    "Idade (meses) início dos sintomas": "intervalo_nascimento_sintomas",
    "Data 1ª colono": "data_primeira_colonoscopia",
    "Unnamed: 10": "intervalo_sintomas_primeira_colonoscopia",
    "Classificação": "classificacao",
})

# ordenar tambem a disposicao das colunas 

df = df.drop(index=0).reset_index(drop=True)

colunas_data = ["data_nascimento", "data_primeira_consulta", "data_primeira_colonoscopia"]

for col in colunas_data:
    df[col] = pd.to_datetime(df[col], errors="coerce").dt.strftime("%d/%m/%Y")

df = df.drop(columns=["nome"])

for index,tempo in enumerate(df['intervalo_sintomas_primeira_consulta'].tolist()) : 
     tempo_meses = tempo.replace("meses","")
     df.loc[index,'intervalo_sintomas_primeira_consulta'] = tempo_meses


df["intervalo_sintomas_primeira_consulta"] = pd.to_numeric(
    df["intervalo_sintomas_primeira_consulta"],
    errors="coerce"
)
df["intervalo_sintomas_primeira_consulta"] = (
    df["intervalo_sintomas_primeira_consulta"].astype("Int64")
)



In [5]:
# df.head(5)

In [6]:
# Criar uma nova coluna data inicio sintomas

# DATA INICICO SINTOMAS =. DATA NASCIMENTO + TEMPO EM MESES



## Criando a coluna data_inicio_sintomas

In [8]:
datas = pd.to_datetime(df["data_nascimento"], format="%d/%m/%Y", errors="coerce")
meses = df["intervalo_nascimento_sintomas"]

df["data_inicio_sintomas"] = [
    d + pd.DateOffset(months=int(m)) if pd.notna(d) and pd.notna(m) else pd.NaT
    for d, m in zip(datas, meses)
]
df['data_inicio_sintomas'] = pd.to_datetime(df['data_inicio_sintomas'], errors="coerce").dt.strftime("%d/%m/%Y")

In [9]:
df[['data_nascimento','intervalo_nascimento_sintomas','data_inicio_sintomas']]

,data_nascimento,intervalo_nascimento_sintomas,data_inicio_sintomas
0,27/09/2013,60,27/09/2018
1,25/10/2024,14,25/12/2025
2,01/01/2006,168,01/01/2020
3,19/04/2019,18,19/10/2020
4,18/05/2012,108,18/05/2021
...,...,...,...
96,08/08/2012,96,08/08/2020
97,05/03/2004,84,05/03/2011
98,25/04/2008,144,25/04/2020
99,19/05/2017,47,19/04/2021


In [10]:
# RECRIAR A COLUNA Tempo entre inicio dos sintomas e 1ª colono = DATA 1 COLO - DATA INICICO SINTOMAS
# recriar essa coluna intervalo_sintomas_primeira_colonoscopia

In [11]:
df.head(5)

,classificacao_idade,data_nascimento,data_primeira_consulta,intervalo_sintomas_primeira_consulta,sexo,tipo,perda_peso,intervalo_nascimento_sintomas,data_primeira_colonoscopia,intervalo_sintomas_primeira_colonoscopia,classificacao,data_inicio_sintomas
0,VEOIBD,27/09/2013,02/03/2023,54,F,Enterorragia,não,60,15/07/2019,6 meses,CU,27/09/2018
1,VEOIBD,25/10/2024,09/06/2026,5,F,Enterorragia,Não,14,20/05/2026,4 meses,CNC,25/12/2025
2,DI Ped,01/01/2006,04/05/2021,13,F,Dor abdominal,Não,168,13/04/2020,1 mês,DC,01/01/2020
3,VEOIBD,19/04/2019,03/08/2023,34,F,Diarreia,Não,18,01/05/2022,19 meses,CNC,19/10/2020
4,Início precoce,18/05/2012,19/01/2023,20,F,Febre,sim,108,26/01/2023,30 meses,DC,18/05/2021


In [12]:
# Com base nessas duas colunas criar a nova coluna
# intervalo_primeira_colonoscopia_inicio_sintomas

# df['intervalo_primeira_colonoscopia_inicio_sintomas'] = 
# df['data_primeira_colonoscopia'] - df['data_inicio_sintomas'] 

# df[['data_primeira_colonoscopia','data_inicio_sintomas']]

# resultados = []
for index in range(0,101):   
    
    data_primeira_colonoscopia=df.loc[index,'data_primeira_colonoscopia']
    data_inicio_sintomas =  df.loc[index,'data_inicio_sintomas']
    
    print(data_inicio_sintomas  + ' ---  ' +data_primeira_colonoscopia)
    


27/09/2018 ---  15/07/2019
25/12/2025 ---  20/05/2026
01/01/2020 ---  13/04/2020
19/10/2020 ---  01/05/2022
18/05/2021 ---  26/01/2023
11/06/2014 ---  25/02/2016
30/05/2020 ---  25/10/2022
30/06/2018 ---  14/06/2018
02/06/2014 ---  17/10/2018
01/03/2020 ---  18/06/2020
24/01/2019 ---  19/12/2019
15/12/2021 ---  10/08/2022
04/08/2022 ---  04/09/2022
11/10/2022 ---  22/11/2023
19/12/2022 ---  28/04/2023
10/02/2023 ---  06/11/2023
19/09/2023 ---  17/01/2024
20/09/2021 ---  04/01/2022
30/06/2024 ---  27/08/2025
14/12/2018 ---  01/05/2019
14/09/2023 ---  26/06/2024
12/01/2016 ---  02/09/2019
05/04/2009 ---  20/07/2009
19/05/2014 ---  12/03/2015
12/11/2017 ---  01/06/2018
25/08/2015 ---  01/01/2016
10/08/2017 ---  27/02/2018
09/07/2022 ---  10/11/2022
17/03/2020 ---  13/08/2020
25/05/2021 ---  29/12/2021
19/08/2018 ---  01/12/2020
13/09/2011 ---  08/07/2018
23/07/2023 ---  06/09/2024
17/04/2012 ---  09/12/2013
09/11/2020 ---  02/02/2022
10/06/2021 ---  10/12/2021
25/04/2023 ---  27/11/2023
2

In [43]:
from datetime import datetime

resultados = []
for index in range(len(df)):
    d1_str = df.loc[index, "data_inicio_sintomas"]
    d2_str = df.loc[index, "data_primeira_colonoscopia"]

    # 1) converte para datetime
    try:
        d1 = datetime.strptime(d1_str, "%d/%m/%Y")
        d2 = datetime.strptime(d2_str, "%d/%m/%Y")
    except (ValueError, TypeError):
        resultados.append(None)
        continue

    # 2) valida ordem
    if d2 <= d1:
        resultados.append(None)
        continue

    # 3) calcula meses completos
    meses = (d2.year - d1.year) * 12 + (d2.month - d1.month) - (d2.day < d1.day)
    resultados.append(meses)

df["intervalo_sintomas_primeira_colo2"] = resultados

In [41]:
diff = df[['data_inicio_sintomas','intervalo_sintomas_primeira_colo2','data_primeira_colonoscopia','intervalo_sintomas_primeira_colonoscopia']]
diff.to_csv('diff.csv',index=False)

In [35]:
# df[df['intervalo_sintomas_primeira_colo2'] == 0]

In [45]:
df.to_csv('dataframe_limpo.csv', index=False)